In [1]:
!pip install torchinfo

In [ ]:
# --- 1. Setup & Imports ---
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import copy
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

transform = transforms.ToTensor()

train_dataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

test_dataset = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64, shuffle=False)

print('Importing completed...\n')

In [ ]:
# --- 2. Build Original CNN ---
cnn_model = nn.Sequential(
    nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(in_features=64*7*7, out_features=128),
    nn.ReLU(),
    nn.Linear(in_features=128, out_features=10)
).to(device)

print('Model understood...')

In [ ]:
# --- 3. Training Loop ---
loss_fn = nn.CrossEntropyLoss()
cnn_optimizer = torch.optim.Adam(cnn_model.parameters(), lr=0.001)
epochs = 5

print("Training in progress...")
for epoch in range(epochs):
    running_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = cnn_model(images)
        loss = loss_fn(outputs, labels)

        cnn_optimizer.zero_grad()
        loss.backward()
        cnn_optimizer.step()

        running_loss += loss.item()

    average_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{epochs} | Error Rate (Loss): {average_loss:.4f}")
print("Training Complete...\n")

In [ ]:
# --- 4. Single Image Test ---
images, real_answer = test_dataset[5]
with torch.no_grad():
    preds = cnn_model(images.unsqueeze(0).to(device))
answer = torch.argmax(preds).item()

plt.imshow(images.squeeze().cpu(), cmap='grey')
plt.title(f"AI Guessed: {answer}  |  Real Answer: {real_answer}")
plt.axis('off')
plt.show()

print("Single test for reference complete...\n")

In [ ]:
# --- 5. Original Model Evaluation ---
cnn_model.eval()
correct_guesses = 0
total_images = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = cnn_model(images)
        _, preds = torch.max(outputs, dim=1)

        total_images += labels.size(0)
        correct_guesses += (preds == labels).sum().item()

cnn_accuracy = 100 * correct_guesses / total_images

print(f"Final Score: The CNN guessed correctly {correct_guesses} out of {total_images} times.")
print(f"CNN Accuracy: {cnn_accuracy:.2f}%\n")

In [ ]:
total_params = sum(p.numel() for p in cnn_model.parameters())
trainable_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

In [ ]:
summary(cnn_model, input_size=(1, 1, 28, 28), col_names=("num_params", "mult_adds"))

In [ ]:
# --- 6. ARSVD Dummy Test ---
def arsvd(W, tau=0.9):
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    p = S / torch.sum(S)
    epsilon = 1e-9
    H_total = -torch.sum(p * torch.log(p + epsilon))
    H_cummulated = -torch.cumsum(p * torch.log(p + epsilon), dim=0)
    target = tau * H_total
    k = torch.where(H_cummulated >= target)[0][0].item() + 1

    U_k = U[:, :k]
    S_k = torch.diag(S[:k])
    Vh_k = Vh[:k,:]

    W_comp = U_k @ S_k @ Vh_k
    return W_comp, k

print("ArSVD computed")

In [ ]:
import torch
torch.manual_seed(42)
base_pattern = torch.randn(5, 2) @ torch.randn(2, 5)
noise = 0.1 * torch.randn(5, 5)
W_test = base_pattern + noise
print(f"test pattern shape: {W_test.shape}")

In [ ]:
thresholds = [0.99, 0.90, 0.50]
for tau in thresholds:
    W_comp, k = arsvd(W_test, tau=tau)
    original_param = W_test.numel()
    new_param = (W_test.shape[0] * k) + (k * W_test.shape[1])
    error = torch.nn.functional.mse_loss(W_test, W_comp)

    print(f"\n--- Threshold (tau) = {tau * 100}% ---")
    print(f"Chosen Rank (k): {k} (out of 5)")
    print(f"Parameters: {new_param} / {original_param}")
    print(f"Information Loss (Error): {error.item():.4f}")
print("\n")

In [ ]:
!pip install torchinfo

In [ ]:
# --- Compressing Convolutional Layer ---
def arsvd_conv_layer(conv_layer, tau=0.90):
  W = conv_layer.weight.data
  in_c, out_c, k_h, k_w = W.shape

  W_2d = W.view(out_c, -1)
  U, S, Vh = torch.linalg.svd(W_2d, full_matrices = False)
  p = S/torch.sum(S)
  epsilon =1e-9
  H_total = -torch.sum(p * torch.log(P + epsilon))
  H_cummulated = -torch.cumsum(p * torch.log(p + epsilon), dim=0)
  target = tau * H_total
  k = torch.where(H_cummulated >= target)[0][0].item() + 1

  ConvA = nn.Con2d(in_channels = in_c, out_channels = k, kernel_size = (k_h, k_w), stride = conv_layer.stride, padding = conv_layer.padding, bias = False)

  Vh_k = Vh[:k, :]
  ConvA.weight.data = Vh_k.view(k, in_c, k_h, k_w)

  ConvB = nn.Conv2d(k, out_channel = out_c, kernal_size=1, bias = True)
  U_k = U[:, :k]
  S_k = torch.diag(S[:k])
  US_k = U_k @ S_k

  ConvB.weight.data = US_k.view(out_c, k, 1, 1)

  if conv_layer.bias is not None:
    ConvB.bias.data = conv_layer.bias.data
  print(f" -> Kept {tau*100}% of info | Auto-selected Rank: {k}")
  return nn.Sequential(ConvA, ConvB)

In [ ]:
# --- 7. Full Connected Layer Compression ---
def arsvd_lin_layer(linear_layer, tau=0.95):
    W = linear_layer.weight.data
    U, S, Vh = torch.linalg.svd(W, full_matrices=False)
    p = S / torch.sum(S)
    epsilon = 1e-9
    H_total = -torch.sum(p * torch.log(p + epsilon))
    H_cummulated = -torch.cumsum(p * torch.log(p + epsilon), dim=0)
    target = tau * H_total
    k = torch.where(H_cummulated >= target)[0][0].item() + 1

    linear_A = nn.Linear(linear_layer.in_features, k, bias=False)
    linear_A.weight.data = Vh[:k, :]

    linear_B = nn.Linear(k, linear_layer.out_features, bias=True)

    U_k = U[:, :k]
    S_k = torch.diag(S[:k])
    linear_B.weight.data = U_k @ S_k

    if linear_layer.bias is not None:
        linear_B.bias.data = linear_layer.bias.data

    print(f" -> Kept {tau*100}% of info | Auto-selected Rank: {k}")
    return nn.Sequential(linear_A, linear_B)

In [ ]:
# --- Implementing ARSVD in CNN ---
print("Starting Adaptive-Rank Neural Network Compression...\n")

comp_cnn = copy.deepcopy(cnn_model)
new_layer = []

for index, layer in enumerate(comp_cnn):
    if isinstance(layer, nn.Linear) and index == 7:
        print(f"Compressing Layer {index}: {layer.in_features}x{layer.out_features}")
        comp_layer = arsvd_lin_layer(layer, tau=0.90)
        new_layer.append(comp_layer)
    elif isinstance(layer, nn.Conv2d) and index == 4:
      print(f"Compressing Layer {index}: {layer.in_features}x{layer.out_features}")
      comp_layer = arsvd_conv_layer(layer, tau=0.90)
      new_layer.append(comp_layer)
    else:
        new_layer.append(layer)

final_comp_cnn = nn.Sequential(*new_layer).to(device)
print("\nCompression Complete!")

total_param_original = sum(p.numel() for p in cnn_model.parameters())
total_param_comp = sum(p.numel() for p in final_comp_cnn.parameters())
print("\n--- Hardware Efficiency Results ---")
print(f"Original Parameters:   {total_param_original:,}")
print(f"Compressed Parameters: {total_param_comp:,}")

percent_saved = 100 * (1 - (total_param_comp / total_param_original))
print(f"Total Model Reduction: {percent_saved:.2f}% smaller!")

print("\n--- Final Compressed Model Summary ---")
summary(final_comp_cnn, input_size=(1, 1, 28, 28), col_names=("num_params", "mult_adds"), device=device)

In [ ]:
# --- 8. Compressed Model Evaluation & Final Comparison ---
final_comp_cnn.eval()

correct_guesses = 0
total_images = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = final_comp_cnn(images)
        _, preds = torch.max(outputs, dim=1)

        total_images += labels.size(0)
        correct_guesses += (preds == labels).sum().item()

comp_cnn_accuracy = 100 * correct_guesses / total_images

orig_stats = summary(cnn_model, input_size=(1, 1, 28, 28), verbose=0)
comp_stats = summary(final_comp_cnn, input_size=(1, 1, 28, 28), verbose=0)

orig_flops = orig_stats.total_mult_adds
comp_flops = comp_stats.total_mult_adds
orig_params = orig_stats.total_params
comp_params = comp_stats.total_params

print("\n================ FINAL COMPARISON ================")
print(f"Original Accuracy:   {cnn_accuracy:.2f}%")
print(f"Compressed Accuracy: {comp_cnn_accuracy:.2f}%")
print(f"-> Accuracy Drop:    {cnn_accuracy - comp_cnn_accuracy:.2f}%\n")

print(f"Original Parameters:   {orig_params:,}")
print(f"Compressed Parameters: {comp_params:,}")
print(f"-> Memory Saved:       {100 * (1 - comp_params / orig_params):.2f}%\n")

print(f"Original FLOPs (MACs):   {orig_flops:,}")
print(f"Compressed FLOPs (MACs): {comp_flops:,}")
print(f"-> Compute Saved:        {100 * (1 - comp_flops / orig_flops):.2f}%")
print("=============================================================")

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np

print("Running final speed benchmarks...")

# --- 1. Measure Inference Time ---
cnn_model.eval()
final_comp_cnn.eval()

# Time the Original Model
start_time = time.time()
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        _ = cnn_model(images)
original_time = time.time() - start_time

# Time the Compressed Model
start_time = time.time()
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        _ = final_comp_cnn(images)
compressed_time = time.time() - start_time

print(f"Original Time: {original_time:.4f}s | Compressed Time: {compressed_time:.4f}s\n")


# --- 2. Plotting the Data ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('ARSVD Compression Results on Custom MNIST CNN', fontsize=18, fontweight='bold')

labels = ['Original CNN', 'ARSVD CNN']
colors = ['#A9C4D7', '#5E8B9E']

# Graph 1: Parameter Reduction
params_data = [orig_params, comp_params]
axes[0].bar(labels, params_data, color=colors, width=0.5)
axes[0].set_title('Parameter Reduction (Complexity)', fontsize=14)
axes[0].set_ylabel('Total Parameters')

for i, v in enumerate(params_data):
    axes[0].text(i, v + (orig_params*0.01), f"{v:,}", ha='center', fontweight='bold')

# Graph 2: Accuracy Preservation (Performance)
acc_data = [cnn_accuracy / 100, comp_cnn_accuracy / 100]
axes[1].bar(labels, acc_data, color=colors, width=0.5)
axes[1].set_title('Accuracy / F1 Score', fontsize=14)
axes[1].set_ylabel('Normalized Accuracy (0 to 1)')
axes[1].set_ylim(0.9, 1.0)
for i, v in enumerate(acc_data):
    axes[1].text(i, v + 0.001, f"{v:.4f}", ha='center', fontweight='bold')

# Graph 3: Inference Time (Speed)
time_data = [original_time, compressed_time]
axes[2].bar(labels, time_data, color=colors, width=0.5)
axes[2].set_title('Total Inference Time (10k Images)', fontsize=14)
axes[2].set_ylabel('Time in Seconds')
for i, v in enumerate(time_data):
    axes[2].text(i, v + (original_time*0.01), f"{v:.4f}s", ha='center', fontweight='bold')

# Clean up layout and display
plt.tight_layout()
plt.subplots_adjust(top=0.88)
plt.show()

In [ ]:
# --- 9. Final Visual Proof ---
import matplotlib.pyplot as plt
import random

print("--- Single Image Test: Original vs Compressed ---")

random_index = random.randint(0, len(test_dataset) - 1)
images, real_answer = test_dataset[random_index]

with torch.no_grad():
    image_tensor = images.unsqueeze(0).to(device)

    #Original Model
    outputs_orig = cnn_model(image_tensor)
    answer_orig = torch.argmax(outputs_orig).item()

    #Compressed Model
    outputs_comp = final_comp_cnn(image_tensor)
    answer_comp = torch.argmax(outputs_comp).item()

# --- Plotting the Results ---
plt.figure(figsize=(6, 6))

plt.imshow(images.squeeze().cpu(), cmap='grey')

title_text = (f"Real Answer: {real_answer}\n\n"
              f"Original CNN Guessed: {answer_orig}\n"
              f"ARSVD CNN Guessed: {answer_comp}")

plt.title(title_text, fontsize=14, fontweight='bold', pad=15)
plt.axis('off')
plt.tight_layout()
plt.show()